# Analysis – SWR-Mediated Memory Reactivation
**Michon Linde et al., Nature Communications**  
*"The Intermediate Hippocampus Integrates Shock-Observation and Spatial Information during Observational Fear Memory"*

Covers: **Fig. 5b–e** · **Extended Data Fig. 5d–f**

> Set `Folder_path` below to the directory containing the summary data tables.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd

import pingouin as pg

import seaborn as sns
from matplotlib import pyplot as plt

## Data loading

In [ ]:
# ── Set this path to the folder containing the summary data tables ──────────
Folder_path = "/data07/Fred/Ctx_Hpc/summaries/NatCom/"

# Explained variance (EV) and reverse explained variance (REV) per session × direction
df_ExpVar = pd.read_parquet(os.path.join(Folder_path, 'table_EV.parquet'))

# EV/REV split by SWR-originating hippocampal subregion (dCA1 / iCA1 / vCA1)
df_ExpVar_SWRarea = pd.read_parquet(os.path.join(Folder_path, 'table_EV_SWRarea.parquet'))

# EV/REV split by the subregion of the reactivated neuron pairs (dorsal / intermediate / ventral)
df_ExpVar_cellarea = pd.read_parquet(os.path.join(Folder_path, 'table_EV_cellarea.parquet'))

# SWR rate per subregion × sleep phase (pre-sleep / post-sleep)
df_swr = pd.read_parquet(os.path.join(Folder_path, 'table_SWR.parquet'))

# Per-unit SWR-triggered firing rate per subregion × sleep phase
df_SWRu = pd.read_parquet(os.path.join(Folder_path, 'table_SWRu.parquet'))

# Pairwise unit–unit co-activity correlations during SWRs (pre-sleep vs. post-sleep)
df_corr = pd.read_parquet(os.path.join(Folder_path, 'table_SWRu_corr.parquet'))

print("Loaded SWR reactivation tables:")
print(f"  df_ExpVar          : {df_ExpVar.shape[0]} rows — EV/REV per session")
print(f"  df_ExpVar_SWRarea  : {df_ExpVar_SWRarea.shape[0]} rows — EV/REV by SWR origin subregion")
print(f"  df_ExpVar_cellarea : {df_ExpVar_cellarea.shape[0]} rows — EV/REV by neuron-pair subregion")
print(f"  df_swr             : {df_swr.shape[0]} rows — SWR rates")
print(f"  df_SWRu            : {df_SWRu.shape[0]} rows — per-unit SWR firing rates")
print(f"  df_corr            : {df_corr.shape[0]} rows — pairwise SWR co-activity correlations")

---
## Figure 5b
**Post-sleep SWRs selectively reactivate the shock-context spatial map.**  
Explained variance (EV, forward replay) and reverse explained variance (REV, control)  
computed from post-sleep SWRs against spatial rate maps estimated in each context  
(safe, shock) and during shock observation.  
EV > REV indicates content-specific reactivation.

In [ ]:
df_ev = df_ExpVar.query("phase == 'full_safe' or phase == 'full_shock' or phase == 'shock_observation'")

fig, ax = plt.subplots(1, 1, figsize=(4, 4))

# Paired animal lines for each phase
for phase, x_pos in [('full_safe', [-0.25, 0.25]),
                      ('full_shock', [0.75, 1.25]),
                      ('shock_observation', [1.75, 2.25])]:
    for pair in np.array(df_ev.query("phase == '{}'".format(phase))
                               .set_index(['rat', 'direction'])
                               .unstack()['explained_variance']):
        ax.plot(x_pos, pair, color='grey', lw=1.0, alpha=0.5)

sns.boxplot(y='explained_variance', x='phase', hue='direction',
            palette='Set1',
            order=['full_safe', 'full_shock', 'shock_observation'],
            boxprops=dict(alpha=0.4), fliersize=0.0,
            data=df_ev, ax=ax)
sns.stripplot(y='explained_variance', x='phase', hue='direction',
              palette='Set1',
              order=['full_safe', 'full_shock', 'shock_observation'],
              size=10, alpha=0.6, dodge=True,
              data=df_ev, ax=ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['EV (forward)', 'REV (reverse control)'], frameon=False)
ax.set(ylabel='explained variance (EV / REV)',
       xlabel='',
       xticklabels=['Safe context', 'Shock context', 'Shock observation'],
       ylim=(0, 1))
sns.despine(offset=True, trim=True)

### Statistics — Fig. 5b | Paired t-tests: EV vs. REV per phase; two-way ANOVA across phases

In [ ]:
# Paired t-test: EV vs. REV within each phase
print("Fig. 5b — Paired t-tests: EV vs. REV within each experimental phase")
for phase in np.unique(df_ev.phase):
    results = pg.ttest(
        df_ev.query("phase == '{}' and direction == 'forward'".format(phase))['explained_variance'],
        df_ev.query("phase == '{}' and direction == 'reverse'".format(phase))['explained_variance'],
        paired=True)
    print("  Phase: {}".format(phase))
    print(results.to_string())
    print()

# Two-way ANOVA: EV ~ direction × phase
var, factor1, factor2 = 'explained_variance', 'direction', 'phase'
data = df_ev[[var, factor1, factor2]].copy()
data['group_combined'] = data[factor1] + '_' + data[factor2]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Fig. 5b — Two-way ANOVA: EV ~ direction × phase")
print(results.to_string())
print()
print(posthoc1.to_string())

---
## Figure 5c
**Shock-context map reactivation is driven by intermediate and ventral CA1 SWRs.**  
Forward EV for the safe and shock context estimated separately for SWRs  
originating in dCA1, iCA1 or vCA1.

In [ ]:
df_ev_swr = df_ExpVar_SWRarea.query("phase == 'full_safe' or phase == 'full_shock'").dropna()

for area in ['dCA1', 'iCA1', 'vCA1']:
    fig, ax = plt.subplots(1, 1, figsize=(1.5, 4))

    # Paired animal lines
    for pair in np.array(
            df_ev_swr.query("direction == 'forward' and area == '{}'".format(area))
                     .set_index(['rat', 'phase'])
                     .unstack()['explained_variance']):
        ax.plot([0, 1], pair, color='grey', lw=1.0, alpha=0.5)

    sns.boxplot(y='explained_variance', x='phase',
                order=['full_safe', 'full_shock'],
                palette=['darkgreen', 'rebeccapurple'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=df_ev_swr.query("direction == 'forward' and area == '{}'".format(area)), ax=ax)
    sns.stripplot(y='explained_variance', x='phase',
                  order=['full_safe', 'full_shock'],
                  palette=['darkgreen', 'rebeccapurple'],
                  size=10, alpha=0.6, dodge=False,
                  data=df_ev_swr.query("direction == 'forward' and area == '{}'".format(area)), ax=ax)

    ax.set(ylabel='explained variance (EV)',
           xlabel='', xticklabels=['Safe', 'Shock'],
           ylim=(0, 1), title='{} SWRs'.format(area))
    sns.despine(offset=True, trim=True)

### Statistics — Fig. 5c | Paired t-tests: EV safe vs. shock context, per SWR-originating subregion

In [ ]:
df_ev_swr = df_ExpVar_SWRarea.query("phase == 'full_safe' or phase == 'full_shock'")

print("Fig. 5c — Paired t-tests: EV safe vs. shock context, by SWR-originating subregion")
for area in np.unique(df_ev_swr.area):
    results = pg.ttest(
        df_ev_swr.query("phase == 'full_safe'  and direction == 'forward' and area == '{}'".format(area))['explained_variance'],
        df_ev_swr.query("phase == 'full_shock' and direction == 'forward' and area == '{}'".format(area))['explained_variance'],
        paired=True)
    print("  SWR subregion: {}".format(area))
    print(results.to_string())
    print()

---
## Figure 5d
**Shock-context map reactivation is driven by intermediate and ventral CA1 neuron pairs.**  
Forward EV for the safe and shock context estimated from neuron pairs  
located in the dorsal, intermediate or ventral hippocampus.

In [ ]:
df_ev_cell = df_ExpVar_cellarea.query("phase == 'full_safe' or phase == 'full_shock'").dropna()

for area in ['dorsal', 'intermediate', 'ventral']:
    fig, ax = plt.subplots(1, 1, figsize=(1.5, 4))

    for pair in np.array(
            df_ev_cell.query("direction == 'forward' and area == '{}'".format(area))
                      .set_index(['rat', 'phase'])
                      .unstack()['explained_variance']):
        ax.plot([0, 1], pair, color='grey', lw=1.0, alpha=0.5)

    sns.boxplot(y='explained_variance', x='phase',
                order=['full_safe', 'full_shock'],
                palette=['darkgreen', 'rebeccapurple'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=df_ev_cell.query("direction == 'forward' and area == '{}'".format(area)), ax=ax)
    sns.stripplot(y='explained_variance', x='phase',
                  order=['full_safe', 'full_shock'],
                  palette=['darkgreen', 'rebeccapurple'],
                  size=10, alpha=0.6, dodge=False,
                  data=df_ev_cell.query("direction == 'forward' and area == '{}'".format(area)), ax=ax)

    ax.set(ylabel='explained variance (EV)',
           xlabel='', xticklabels=['Safe', 'Shock'],
           ylim=(0, 1), title='{} hippocampus'.format(area))
    sns.despine(offset=True, trim=True)

---
## Figure 5e
**Shock-context reactivation gain (Δ EV) is greater in RECALLER animals — intermediate and ventral hippocampus.**  
ΔDEV = EV(shock context) − EV(safe context) per animal, split by recall status (NON-RECALLER / RECALLER),  
separately for each hippocampal subregion.

In [ ]:
df_ev_cell = df_ExpVar_cellarea.query("phase == 'full_safe' or phase == 'full_shock'")

tmp_ev = (df_ev_cell.query("direction == 'forward'")
          .set_index(['rat', 'area', 'phase'])
          .unstack())
tmp_ev['delta']    = np.diff(tmp_ev.explained_variance)        # EV(shock) − EV(safe)
tmp_ev['learning'] = tmp_ev['context_learning', 'full_shock']
tmp_ev = tmp_ev[['delta', 'learning']].reset_index().droplevel(1, axis=1)

for area in ['dorsal', 'intermediate', 'ventral']:
    fig, ax = plt.subplots(1, 1, figsize=(1.5, 4))
    sns.boxplot(y='delta', x='learning',
                order=['not learned', 'learned'],
                palette=['grey', 'orchid'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=tmp_ev.query("area == '{}'".format(area)), ax=ax)
    sns.stripplot(y='delta', x='learning',
                  order=['not learned', 'learned'],
                  palette=['grey', 'orchid'],
                  size=10, alpha=0.6, dodge=False,
                  data=tmp_ev.query("area == '{}'".format(area)), ax=ax)

    ax.axhline(0, ls='--', lw=0.5, color='grey')
    ax.set(ylabel='ΔEV (shock − safe)',
           xlabel='',
           xticklabels=['NON-RECALLER', 'RECALLER'],
           ylim=(-0.2, 0.6),
           title='{} hippocampus'.format(area))
    sns.despine(offset=True, trim=True)

### Statistics — Fig. 5d–e | Paired t-tests: EV safe vs. shock; independent t-tests: ΔEV by recall status

In [ ]:
df_ev_cell = df_ExpVar_cellarea.query("phase == 'full_safe' or phase == 'full_shock'")

# Fig. 5d: paired t-test, EV safe vs. shock per neuron-pair subregion
print("Fig. 5d — Paired t-tests: EV safe vs. shock context, by neuron-pair subregion")
for area in np.unique(df_ev_cell.area):
    results = pg.ttest(
        df_ev_cell.query("phase == 'full_safe'  and direction == 'forward' and area == '{}'".format(area))['explained_variance'],
        df_ev_cell.query("phase == 'full_shock' and direction == 'forward' and area == '{}'".format(area))['explained_variance'],
        paired=True)
    print("  Subregion: {}".format(area))
    print(results.to_string())
    print()

# Fig. 5e: independent t-test, ΔEV by recall status per subregion
tmp_ev = (df_ev_cell.query("direction == 'forward'")
          .set_index(['rat', 'area', 'phase'])
          .unstack())
tmp_ev['delta']    = np.diff(tmp_ev.explained_variance)
tmp_ev['learning'] = tmp_ev['context_learning', 'full_shock']
tmp_ev = tmp_ev[['delta', 'learning']].reset_index().droplevel(1, axis=1)

print("Fig. 5e — Independent t-tests: ΔEV (shock − safe) by recall status")
for area in np.unique(tmp_ev.area):
    results = pg.ttest(
        tmp_ev.query("learning == 'not learned' and area == '{}'".format(area))['delta'],
        tmp_ev.query("learning == 'learned'     and area == '{}'".format(area))['delta'],
        paired=False)
    print("  Subregion: {}".format(area))
    print(results.to_string())
    print()

---
## Extended Data Figure 5d
**No change in SWR rate between pre- and post-sleep across hippocampal subregions.**  
SWR rate (Hz) in dCA1, iCA1 and vCA1 during pre-sleep and post-sleep sessions.

In [ ]:
plt.rc('xtick', labelsize=12)

tmp_plot = df_swr.query("phase != 'delta'")

fig, ax = plt.subplots(1, 1, figsize=(2, 3))
var = 'swr_rate'

sns.boxplot(x='area', hue='phase', y=var, data=tmp_plot,
            order=['dCA1', 'iCA1', 'vCA1'],
            hue_order=['presleep', 'postsleep'],
            palette=['lightgray', 'black'],
            boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax)
sns.stripplot(x='area', hue='phase', y=var, data=tmp_plot,
              order=['dCA1', 'iCA1', 'vCA1'],
              hue_order=['presleep', 'postsleep'],
              palette=['lightgray', 'black'],
              dodge=True, size=8, alpha=0.5, ax=ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['pre-sleep', 'post-sleep'], frameon=False)
ax.set(ylim=(0, 0.6),
       ylabel='SWR rate (Hz)',
       xlabel='')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 5d | Two-way ANOVA: SWR rate ~ subregion × sleep phase

In [ ]:
tmp_plot = df_swr.query("phase != 'delta'")
var, factor1, factor2 = 'swr_rate', 'area', 'phase'
data = tmp_plot[[var, factor1, factor2]].copy()
data['group_combined'] = data[factor1] + '_' + data[factor2]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Ext. Data Fig. 5d — Two-way ANOVA: SWR rate ~ subregion × sleep phase")
print(results.to_string())
print()
print(posthoc1.to_string())

---
## Extended Data Figure 5e
**No change in per-unit SWR-triggered firing rate between pre- and post-sleep.**  
Mean firing rate of putative pyramidal neurons during SWRs detected in any subregion (all),  
per hippocampal subregion of the recorded neuron, pre- vs. post-sleep.

In [ ]:
plt.rc('xtick', labelsize=12)

tmp_plot = df_SWRu.query(
    "SWR_area == 'all' and neuron_type == 'pyr' and phase != 'delta' and cluster_pole != 'neither'")

fig, ax = plt.subplots(1, 1, figsize=(2, 3))
var = 'mean_SWR_rate'

sns.boxplot(x='cluster_pole', hue='phase', y=var, data=tmp_plot,
            hue_order=['presleep', 'postsleep'],
            palette=['lightgray', 'black'],
            boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax)
sns.stripplot(x='cluster_pole', hue='phase', y=var, data=tmp_plot,
              hue_order=['presleep', 'postsleep'],
              palette=['lightgray', 'black'],
              dodge=True, size=8, alpha=0.15, ax=ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['pre-sleep', 'post-sleep'], frameon=False)
ax.set(ylim=(0, 65),
       xticklabels=['D', 'I', 'V'],
       ylabel='mean SWR firing rate (Hz)',
       xlabel='')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 5e | Two-way ANOVA: per-unit SWR firing rate ~ subregion × sleep phase

In [ ]:
var, factor1, factor2 = 'mean_SWR_rate', 'cluster_pole', 'phase'
data = tmp_plot[[var, factor1, factor2]].copy()
data['group_combined'] = data[factor1] + '_' + data[factor2]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Ext. Data Fig. 5e — Two-way ANOVA: per-unit SWR firing rate ~ subregion × sleep phase")
print(results.to_string())
print()
print(posthoc1.to_string())

---
## Extended Data Figure 5f
**No change in pairwise SWR co-activity correlations between pre- and post-sleep.**  
Mean Pearson correlation of SWR-triggered population vectors between pairs of neurons  
recorded in the same subregion (dorsal–dorsal, intermediate–intermediate, ventral–ventral),  
pre- vs. post-sleep.

In [ ]:
plt.rc('xtick', labelsize=12)

tmp_plot = df_corr.query(
    "(pole == 'dorsal-dorsal' or pole == 'intermediate-intermediate' or pole == 'ventral-ventral')"
    " and (phase == 'presleep' or phase == 'postsleep')")

fig, ax = plt.subplots(1, 1, figsize=(2, 3))
var = 'correlation'

sns.boxplot(x='pole', hue='phase', y=var, data=tmp_plot,
            order=['dorsal-dorsal', 'intermediate-intermediate', 'ventral-ventral'],
            hue_order=['presleep', 'postsleep'],
            palette=['lightgray', 'black'],
            boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax)
sns.stripplot(x='pole', hue='phase', y=var, data=tmp_plot,
              order=['dorsal-dorsal', 'intermediate-intermediate', 'ventral-ventral'],
              hue_order=['presleep', 'postsleep'],
              palette=['lightgray', 'black'],
              dodge=True, size=8, alpha=0.5, ax=ax)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:2], ['pre-sleep', 'post-sleep'], frameon=False)
ax.set(ylim=(-0.2, 0.2),
       xticklabels=['D–D', 'I–I', 'V–V'],
       ylabel='SWR co-activity correlation (r)',
       xlabel='')
sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 5f | Two-way ANOVA: SWR co-activity ~ subregion × sleep phase

In [ ]:
var, factor1, factor2 = 'correlation', 'phase', 'pole'
data = tmp_plot[[var, factor1, factor2]].copy()
data['group_combined'] = data[factor1] + '_' + data[factor2]

results  = pg.anova(data=data, dv=var, between=[factor1, factor2], ss_type=2)
posthoc1 = pg.pairwise_tests(dv=var, between=[factor1, factor2],
                              padjust='holm', effsize='cohen', data=data)

print("Ext. Data Fig. 5f — Two-way ANOVA: SWR co-activity ~ subregion × sleep phase")
print(results.to_string())
print()
print(posthoc1.to_string())